[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slp-hu/class-PythonDS/blob/main/introDA_L13_2026.ipynb)

> ☝️ 上のバッジをクリックすると Google Colab で開きます。

# 第13回　本物の個票で確かめる ― 生成AIと勉強時間

**題材：第60回 学生生活実態調査（全国大学生活協同組合連合会／SSJDA 1794）**

第11回は**擬似**ミクロデータで「個票の扱い方」を、第12回は県単位の集計で「**探索と確証**」を学びました。
今回は **本物の個票（回答者 25,340 人）** です。しかも問いは、あなた自身に関わることです。

> ## 🔍 今日の問い
> **「生成AIを使う学生は、勉強時間が短いのだろうか？」**
>
> 素朴には「AIに任せるから勉強しなくなる（＝負の関係）」と思うかもしれません。データはどう答えるでしょう。

---

### 📋 前回の講評から ― 今日ずっと意識する3つの約束

確認テストと課題11の講評で見えた、みんなのつまずきです。今日はこれを**手を動かして直します**。

| 約束 | 前回なにが起きたか | 今日どこで練習するか |
|---|---|---|
| **① 相関や差は、まず「符号（向き）」を読む** | 相関 r の誤答の大半が**符号エラー**（負だと思い込んだ）。栃木県も「男女で向きが逆」を12/13名が見落とし | §5・§6（**符号が逆転する集団**が出てきます） |
| **② 率を出したら「母数は誰か」を必ず書く** | 演習1で母数を「雇用者」と取り違える解答 | §4（母数を変えると 28.1% → 51.5% に化けます） |
| **③ 数字は「出典」とセットで書く** | 「37」と「41」の取り違え。記述の減点はほぼ**出典の欠落** | §0・演習3（DOI まで書きます） |

そして第11回の演習で最も多かった △ は、**「数値は出したが、読み取りを言葉で書いていない」**（24名）でした。
今日は各節に「**✍️ 読み取りを書く**」欄があります。**必ず日本語の文で**書いてください。


## 0. このデータの約束 ― 読んでから始める ⚠️

今日のデータは、これまでと決定的に違います。**擬似データではなく、実在の学生 25,340 人が答えた本物の個票**です。
だから、扱いにも約束があります。

**出どころ**
- 全国大学生活協同組合連合会「第60回 学生生活実態調査」
- 東京大学社会科学研究所附属社会調査・データアーカイブ研究センター（SSJDA）から、**この授業のために利用許諾を得て**提供されています。
- 二次分析の結果を書くときは、**出典を明記**します（→ 演習3）。DOI: `https://doi.org/10.34500/SSJDA.1794`

**やってはいけないこと（利用許諾の条件）**
1. **再配布しない**（友人に送る、SNS やリポジトリに上げる、公開の GitHub に置く、すべて不可）。
2. **授業の目的以外に使わない**。
3. **生成AI に渡さない**。ChatGPT などのチャット欄に貼り付けたり、ファイルとして添付したりしない。
   （今日の分析テーマがまさに生成AIなのは、皮肉ですが、良い機会です。「便利な道具に**何を渡してよいか**」は、これからのあなたの職業倫理そのものです。
   なお、分析用の Colab にアップロードすることと、生成AI に中身を読ませることは、**まったく別のこと**です。）
4. 授業で使い終わったら、指示に従って**手元と Colab のコピーを消す**。

> 📌 このデータは**公開サイトから各自で取得するものではありません**（第11回・第12回とはそこが違います）。
> **学内の LMS（Moodle）にある授業ページからダウンロード**してください。LMS の外にコピーを作らないこと。
> うまく取得できないときは**申し出てください**。

**Colab で使ってよいか？** ── **はい、この授業では Colab へのアップロードは認められています**（§準備の手順で左の「ファイル」に置きます）。
ただし、次の区別を必ず守ってください。

- ✅ **よい**：Colab のセッションに一時的にアップロードして分析する。
- ❌ **だめ**：GitHub など**公開の場所**に置く。共有リンクで誰でも見られる状態にする。
- ❌ **だめ**：ChatGPT などの**生成AIのチャット欄に貼り付ける／ファイルを添付する**。

**ファイル**
- `1794.csv` … 本体（1行＝1人の回答）
- `1794_label.txt` … **コードブック**（変数名と選択肢の意味）
- `1794_readme.txt` … 提供者からの注意書き（**今日はこれが主役級に重要**）


## 準備（まず下のセルを実行）

> 📂 **Colab でのファイルの置き方**
> Colab は、**最初にセルを1つ実行してランタイムに接続してからでないと、ファイルをアップロードできません。**
> 1. **まず下の準備セルを実行**します。これで**ランタイムが接続**されます。
> 2. 接続できたら、画面**左の「ファイル」📁**に `1794.csv` と `1794_label.txt` を置きます。
> 3. そのあと **§1 以降**を上から実行します。


In [ ]:
# ① まずこのセルを実行（→ランタイムに接続）→ 左の「ファイル」📁 にデータを置く → ②以降を実行
import warnings
import pandas as pd
import numpy as np
warnings.simplefilter('ignore', category=pd.errors.PerformanceWarning)
try:
    import japanize_matplotlib
except ModuleNotFoundError:
    !pip install -q japanize-matplotlib
    import japanize_matplotlib
import matplotlib.pyplot as plt

pd.set_option('display.width', 120)
PATH  = '1794.csv'
LABEL = '1794_label.txt'
print('準備OK')

## 1. 読み込みと「コードブックを引く」 🖐

### 【やってみよう 1-a】文字コードは、毎回確かめる

第11回の就業構造のCSVは **cp932（Shift-JIS）** でした。「公的統計は cp932」と覚えてしまった人はいませんか。
**このファイルは違います。** データごとに違うので、**毎回確かめる**しかありません。

下のセルの `____` を埋めて読み込もう。うまくいかなければ `'cp932'` と `'utf-8'` の両方を試して、
**エラーにならず、日本語の列名や値が化けない方**を選ぶこと。


In [ ]:
# ヒント：encoding は 'utf-8' か 'cp932'。low_memory=False は列の型推定を安定させるおまじない
df = pd.read_csv(PATH, encoding=____, low_memory=False)
print('行数（回答者数）:', f'{len(df):,}')
print('列数（変数の数）:', df.shape[1])
df.head(3)

上の `df.head(3)` を見てください。列名が `q1d` `q2g` `q10a_1` … と**記号**です。
そして値も `1` `2` `3` … という**符号（コード）**。これだけでは何のことか分かりません。

第11回では `Prefecture`（符号）と `T_Prefecture`（ラベル）が**対**になっていました。
今回は**ラベル列がありません**。代わりに、**別ファイルのコードブック** `1794_label.txt` を引きます。
実データではこちらの方がふつうです。


### 🧩 ちょっと寄り道：自分で「関数」を作る（`def`）

これまで `len(...)` や `df.mean()` のような**用意された関数**を使ってきました。
ここからは、**自分で関数を作って**使います。前回の `lambda`（その場かぎりの短い関数）を、名前つきで少し長く書けるようにしたもの、と思ってください。

`def` の形はこれだけです：

```python
def 名前(受け取るもの):      # ← 「入口」。受け取る値に仮の名前をつける
    ...処理...
    return 返すもの          # ← 「出口」。呼び出し元に返す値
```

たとえば「2つの数を足して返す」関数：

```python
def たす(a, b):
    return a + b

たす(3, 5)   # → 8   （a に 3、b に 5 が入る）
```

ポイントは3つだけ：
- **`def 名前(...):`** で始め、中身は**インデント（字下げ）**して書く。
- カッコの中の名前（`a`, `b`）は**仮の置き場所**。呼ぶときに実際の値が入る。
- **`return`** で値を返す。`return` した値が、呼び出した場所に戻ってくる。

> 💡 なぜ関数にするのか：**同じ処理を何度も使う**とき、1か所書いておけば名前を呼ぶだけで済むからです。
> このあと、コードブックを引く処理や、勉強時間を計算する処理を関数にします（何度も使うので）。


### 【やってみよう 1-b】コードブックを引く関数を使う

下のセルは**実行するだけ**です。中身はさっき説明した `def` で作った関数（`codebook`）で、
**変数名を渡すと、その変数のラベルと選択肢の意味を表示**します。仕組みは今は分からなくてOK、まず使ってみましょう。


In [ ]:
def codebook(var):
    """1794_label.txt から、変数 var の説明と選択肢の意味を表示する"""
    lines = open(LABEL, encoding='utf-8').read().split('\n')
    値ラベル開始 = next(i for i, l in enumerate(lines) if l.startswith('値'))

    # 前半＝変数情報（変数名 / 位置 / ラベル）
    for ln in lines[:値ラベル開始]:
        p = ln.split('\t')
        if len(p) >= 3 and p[0] == var:
            print(f'■ {var} : {p[2]}')
            break

    # 後半＝値ラベル（変数名の行に続いて、変数名が空欄の行が選択肢）
    for i in range(値ラベル開始, len(lines)):
        p = lines[i].split('\t')
        if p[0] == var and len(p) >= 3:
            print(f'   {p[1]} = {p[2]}')
            for j in range(i + 1, len(lines)):
                q = lines[j].split('\t')
                if q[0] != '' or len(q) < 3:
                    break
                print(f'   {q[1]} = {q[2]}')
            return
    print('   （数量変数：選択肢なし）')

codebook('q2g')      # 今日の主役：文章生成系AIの利用
print()
codebook('q10a_1')   # 勉強時間（大学）
print()
codebook('q1d')      # 学部

> 💡 `q2g` の選択肢に注目。**「利用している」と「利用していない」の2択ではありません**。
> 「無料版」「有料版」「過去に使ったが今は使っていない」「今後使いたい」「使うつもりはない」「知らない」の**6つ**。
> あとで「AIを使う人／使わない人」を分けるとき、**この6つのどれを、どちらに入れるか**を自分で決めることになります（→ §5）。


## 2. Weight が無い ― この 25,340 人は「誰」なのか 🖐

第11回では、人口を出すのに **Weight（集計用乗率）を合計**しました。今回はどうでしょう。


In [ ]:
# Weight らしき列があるか探す（実行するだけ）
cand = [c for c in df.columns if 'weight' in c.lower() or 'w' == c.lower() or '乗率' in c]
print('Weight らしき列:', cand if cand else 'なし')
print('行数:', f'{len(df):,}')

**Weight はありません。** では「行数 25,340 = 日本の大学生の数」でしょうか。もちろん違います。

`1794_readme.txt` には、こう書かれています（要点）：

- 報告書は、経年比較のために**指定された30大学生協の回収数**で集計している。
- **本データのケース数は 25,340 で、報告書の数値とは一致しない。**
- どの30生協かは**非公開**。

さらに、そもそもこの調査は**大学生協**を通じた調査です。生協のない大学の学生や、生協を使わない学生は入りにくい。

| | 第11回（就業構造・擬似個票） | 第13回（学生生活実態調査） |
|---|---|---|
| Weight | **ある**（1人が何人分かの倍率） | **ない** |
| 行数の意味 | 標本の大きさ（220,391） | 標本の大きさ（25,340） |
| 母集団の人数 | Weight合計で出せる（約1.1億） | **出せない** |
| 全国に一般化 | 設計上は可能 | **できない**（生協・回収の偏り） |

> 📌 **今日の結論はすべて「この 25,340 人について言えること」です。**
> 「日本の大学生は…」と書いたら、それは言い過ぎ。§8 でもう一度確かめます。
> 第12回の言葉でいえば、これは**カバレッジ（誰が入っていないか）の問題**です。


## 3. 欠損の罠 ― 「無回答」と「非該当」が区別されていない 🖐🧩

第11回で、配偶関係が有業者では全部 NaN になっていた罠を覚えていますか。
「その人には**そもそも聞いていない**（非該当）」というタイプの欠損でした。

今回の `readme` には、もっと厄介なことが書いてあります。

> **無回答と非該当は区別せず、両者ともにシステム欠損値（ブランク）となっています。**

つまり、**空欄を見ても「答えなかった」のか「聞かれなかった」のか分からない**。
この状況で、うっかりやりがちな処理があります。


### 【やってみよう 3-a】勉強時間の欠損を数える

勉強時間は、**時間**と**分**が別の列に入っています（第11回の「多段の見出し」と同じで、実データはこういう形が多い）。

- `q10a_1` / `q10a_2` … 最近1週間の勉強時間（**大学**）の 時間 / 分
- `q10b_1` / `q10b_2` … 同（**大学以外**）の 時間 / 分

まず、それぞれの欠損がいくつあるか数えよう。


In [ ]:
# ヒント：欠損の数は df[列].isna().sum()
for c in ['q10a_1', 'q10a_2', 'q10b_1', 'q10b_2']:
    print(f'{c}: 欠損 {df[c].____().____():,} 件')

### 【やってみよう 3-b】`fillna(0)` はここで何をしてしまうか

「欠損は0にしとけばいいや」――よくやります。でも今回、`q10a_1`（大学での勉強時間）の欠損を 0 にすると、
それは「**この人は大学で0時間しか勉強していない**」と主張したことになります。
本当は「答えなかっただけ」かもしれないのに。

2つの処理を並べて、平均がどれだけ変わるか見てみよう。

> 🧩 下のコードでは、**勉強時間を計算する処理を `時間分` という関数**にしています（同じ計算を「大学」と「大学以外」で2回使うため）。
> `.mask(条件)` は「条件が True の場所を欠損にする」命令です。

In [ ]:
# 【誤】欠損をすべて 0 とみなす
勉強_誤 = (df['q10a_1'].fillna(0) * 60 + df['q10a_2'].fillna(0)) \
        + (df['q10b_1'].fillna(0) * 60 + df['q10b_2'].fillna(0))

# 【正】時間・分の「両方」が欠損なら、その人の値は欠損のまま残す
def 時間分(h, m):
    両方欠損 = df[h].isna() & df[m].isna()
    return (df[h].fillna(0) * 60 + df[m].fillna(0)).mask(両方欠損)

勉強 = 時間分('q10a_1', 'q10a_2') + 時間分(____, ____)   # ← 大学以外の分を足す
df['勉強分'] = 勉強

print(f'【誤】欠損を0に : 平均 {勉強_誤.mean():.1f} 分/週   (n = {勉強_誤.notna().sum():,})')
print(f'【正】欠損を保持 : 平均 {勉強.mean():.1f} 分/週   (n = {勉強.notna().sum():,}, 欠損 {勉強.isna().sum():,})')
print(f'\n平均の差: {勉強_誤.mean() - 勉強.mean():.1f} 分/週')

**約 72 分/週**も違います。欠損 3,503 人を「勉強0分」として平均に混ぜたので、平均が**押し下げられた**わけです。

> 📌 教訓：**`fillna(0)` を書く前に、「その 0 は何を意味するか」を言葉にする。**
> 意味が言えないなら、埋めずに**欠損のまま残して、その人を分析から外す**（＝今日のやり方）。
> そして「**何人を外したか**」を必ず報告する。これも出典と同じで、**読者が検算できるようにするため**です。

**✍️ 読み取りを書く（1行でよい）**：欠損を0にすると平均は上がりますか、下がりますか。それはなぜですか。

> （ここに自分の言葉で書く）


## 4. 多重回答と「母数」 ― 合計が 100% を超える 🖐

`q2h_1` 〜 `q2h_9` は「生成AIの利用目的」で、**あてはまるものをすべて選ぶ**形式（多重回答）です。
`readme` によれば **1＝選択、0＝非選択**。

ここに罠があります。**AIを使っていない人の行にも、0 が入っている**のです。
だから「`q2h_1` の平均」を全員で取ると、それは「全学生のうち、AIを授業や研究に使っている人の割合」になります。
「**AI利用者のうち**、授業や研究に使っている割合」を知りたいなら、**母数を絞らなければいけません**。

前回の講評で「母数を取り違えた」解答がありました。まさにここです。


In [ ]:
print('q2h_1（利用目的：授業や研究）の値:', df['q2h_1'].value_counts().to_dict())
print('→ 0/1 のみ。欠損なし＝非利用者にも 0 が入っている\n')

# 母数A：全員
率_全員 = df['q2h_1'].mean() * 100

# 母数B：現在AIを利用している人（q2g が 1=無料版 または 2=有料版）
利用者 = df[df['q2g'].____([1, 2])]      # ← 「1 か 2 のどれかに含まれる」を選ぶメソッド
率_利用者 = 利用者['q2h_1'].mean() * 100

print(f'母数＝全員 {len(df):,}人           → {率_全員:.1f}%')
print(f'母数＝現在利用者 {len(利用者):,}人   → {率_利用者:.1f}%')

**同じ列から 28.1% と 51.5% という2つの数字が出ました。** どちらも間違いではありません。
**違うことを測っている**だけです。だから、率を書くときは必ず「**〜のうち**」を添えます。

### 【やってみよう 4-b】利用目的の一覧を作る（母数＝現在利用者）


In [ ]:
目的 = {1: '授業や研究', 2: '論文・レポート作成の参考', 3: 'メール等の文章作成',
        4: 'エントリーシート作成の参考', 5: '翻訳・外国語作文', 6: 'プログラミング・Excel関数',
        7: '相談・雑談相手', 8: '遊び・興味', 9: 'その他'}

合計 = 0
for i, name in 目的.items():
    p = 利用者[f'q2h_{i}'].____() * 100      # ← 0/1 の列の「平均」は何を表す？
    合計 += p
    print(f'  {name:22s} {p:5.1f}%')
print(f'\n  {"合計":22s} {合計:5.1f}%   ← 100% を超える！')
print(f'  1人あたりの平均選択数: {利用者[[f"q2h_{i}" for i in range(1,10)]].sum(axis=1).mean():.2f} 個')

合計 **214.3%**。円グラフにしてはいけない理由がこれです（**多重回答は足しても100%にならない**）。
1人が平均 **2.14 個**選んでいるからです。

> 📌 **0/1 の列の平均 ＝ 「1 の割合」**。便利ですが、**誰を母数にしたか**でまったく別の数字になります。

**✍️ 読み取りを書く**：利用目的の上位2つは何ですか。母数を明記して1文で書きなさい。

> （例に倣って自分で書く：「**現在AIを利用している学生 11,231 人のうち**、…」）


## 5. 探索 ― AIを使う学生は、勉強時間が短い？ 🖐

いよいよ今日の問いです。まず**群を定義**します。ここが分析の勝負どころ。

`q2g` は6択でした。今回はこう決めます（**決めたら書く**。読者が再現できるように）：

| 群 | `q2g` の値 | 人数 |
|---|---|---|
| **AI利用** | 1（無料版）, 2（有料版） | これから数える |
| **AI非利用** | 4（今後使いたい）, 5（使うつもりはない） | 同上 |
| **除外** | 3（過去に使ったが今は使っていない）, 6（知らない） | 同上 |

なぜ 3 と 6 を除くのか。**3 はどちらとも言えない**（過去の利用が勉強時間に影響しているかもしれない）。
**6 は「知らない」**という別の状態です。曖昧なものを無理にどちらかへ入れると、結果が定義に振り回されます。

> ⚠️ ただし忘れないこと：**この線引き自体が、あなたの選択**です。線を引き直せば数字は変わります。


In [ ]:
# AI = 1（利用）/ 0（非利用）/ NaN（除外）
df['AI'] = np.where(df['q2g'].isin([1, 2]), 1,
           np.where(df['q2g'].isin([____, ____]), 0, np.nan))   # ← 非利用の2つ

分析 = df[df['AI'].notna() & df['勉強分'].notna()]
print(f'分析対象 n = {len(分析):,}')

g = 分析.groupby('AI')['勉強分'].agg(['count', 'mean', 'median'])
g.index = ['AI非利用', 'AI利用']
print(g.round(1))

素朴な差 = g.loc['AI利用', 'mean'] - g.loc['AI非利用', 'mean']
print(f'\n素朴な差（AI利用 − 非利用）: {素朴な差:+.1f} 分/週  = {素朴な差/60:+.2f} 時間/週')

### 符号を、まず読む

出た数字は **+42.4 分/週**。プラスです。中央値でも +60 分（300分 → 360分）。

素朴な仮説は「AIを使うと勉強しなくなる」＝**マイナス**を予想していました。**符号が逆**です。

前回、相関 r = +0.236 の符号を「負」と答えた人が誤答の大半でした。**予想と逆の符号が出たとき、それが探索で得た事実です。**
自分の予想に合わせて符号を読み替えてはいけません。

> ⚠️ ただし、ここで「**AIを使えば勉強するようになる**」と書いたら、第12回の授業は無駄だったことになります。
> これは**探索**です。次に何を疑うんでしたか。

**✍️ 読み取りを書く**：この結果を、因果を含めずに1文で書きなさい（母数と単位を入れること）。

> （ここに書く）


## 6. 交絡を疑う ― 学部で層別してみる 🖐

第12回で学んだ**交絡**：x（AI利用）と y（勉強時間）の**両方に影響する第3の要因**があると、
x と y が一緒に動いて見える、というものでした。

学生で真っ先に思いつく第3の要因は **学部** です。

- 理工系はプログラミングでAIを使いそう（→ AI利用率が高い）。
- 医歯薬系は実習や国家試験で勉強時間が長そう（→ 勉強時間が長い）。

もしそうなら、AIと勉強時間の関係は「学部の違い」を見ているだけかもしれません。
**学部ごとに分けて、同じ差を計算し直します**（＝層別）。


In [ ]:
学部名 = {1: '文科系', 2: '理工系', 3: '医歯薬系'}

print(f'{"学部":8s} {"n":>7s} {"AI利用率":>9s} {"平均勉強":>9s}   AI利用 − 非利用')
print('-' * 62)
for k, name in 学部名.items():
    g = 分析[分析['q1d'] == k]
    差 = g[g['AI'] == 1]['勉強分'].____() - g[g['AI'] == ____]['勉強分'].mean()   # ← 群ごとの平均の差
    print(f'{name:8s} {len(g):7,} {g["AI"].mean()*100:8.1f}% {g["勉強分"].mean():8.0f}分   {差:+8.1f} 分/週')

### 🚨 符号が逆転する集団がある

| 学部 | AI利用率 | 平均勉強 | AI利用 − 非利用 |
|---|---:|---:|---:|
| 文科系 | 47.8% | 525分 | **+48.1** |
| 理工系 | 67.3% | 627分 | **+52.3** |
| 医歯薬系 | 43.2% | 759分 | **−39.6** ← ! |

**医歯薬系だけ、AIを使う学生の方が勉強時間が短い**のです。全体では +42.4 だったのに。

これは前回の**栃木県**とまったく同じ形です。栃木は「全国では女性が流出」なのに「男性は転入超過・女性は転出超過」で**男女で向きが逆**でした。13名中12名がこれを見落としました。
**全体の符号を、部分にそのまま当てはめてはいけない。**

そして交絡の正体も見えます。方向が**2つ**あります：

- **理工系**：AI利用率が高い（67.3%）**かつ**勉強時間が長い（627分）→ 全体の差を**押し上げる**
- **医歯薬系**：AI利用率が低い（43.2%）**なのに**勉強時間が最長（759分）→ 全体の差を**押し下げる**

つまり全体の +42.4 分は、「AIの効果」と「学部の構成の違い」が**混ざった**数字です。

**✍️ 読み取りを書く**：医歯薬系だけ符号が逆であることを、1〜2文で説明しなさい（「全体では」「しかし」を使うとよい）。

> （ここに書く）


## 7. そろえて比べる ― 層別してから平均する 🖐

第12回の最後にやった**傾向スコア・マッチング**の心は「**似た人どうしを比べる**」でした。
あのときは県（47個の集団）でしたが、**今日は個人単位のデータ**です。第12回で「確証には個人単位データが要る」と言った、その条件がここでは揃っています。

やり方はシンプルにします。**学年・学部・性別・アルバイト有無・住まい**が全部同じ人どうしのグループ（セル）を作り、
**セルの中でだけ AI利用 − 非利用 の差**を取って、最後に人数で重みをつけて平均します（＝**層別**）。
「似た人どうしを比べる」を、そのまま素直に実装したものです。


In [ ]:
共変量 = ['q1e', 'q1d', 'q1g', 'q15a', 'q1i']   # 学年・学部・性別・バイト・住まい
S = 分析.dropna(subset=共変量 + ['勉強分', 'AI']).copy()
print(f'共変量に欠損のある人を外して n = {len(S):,}（{len(分析)-len(S)} 人減）')

素朴 = S[S.AI==1]['勉強分'].mean() - S[S.AI==0]['勉強分'].mean()

分子 = 分母 = 0
for _, g in S.groupby(____):                         # ← 共変量が全部同じ人でグループを作る
    if g['AI'].nunique() == 2:                       # 両方の群がいるセルだけ比較できる
        w = (g['AI'] == 1).sum()                     # 処置群の人数で重みづけ
        分子 += w * (g[g.AI==1]['勉強分'].mean() - g[g.AI==0]['勉強分'].mean())
        分母 += w
層別ATT = 分子 / 分母

print(f'\n素朴な差       : {素朴:+.1f} 分/週')
print(f'層別でそろえた差: {層別ATT:+.1f} 分/週   （比較できたセルが処置群の {分母/(S.AI==1).sum()*100:.1f}% をカバー）')

**+42.1 分 → +27.1 分**。差の**およそ3分の1が、学部などの構成の違いで説明できた**わけです。
そして**ゼロにはならなかった**。

> 💡 §5 の素朴な差は +42.4 分でしたが、ここでは +42.1 分です。**共変量に欠損のある 301 人を外したから**、
> 対象がわずかに変わったのです。数字が動いたら理由を言えること、そして「**何人外したか**」を書くこと。

別のやり方でも確かめましょう（**同じ結論が出るか**＝頑健性の確認）。


In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression

X = pd.get_dummies(S[共変量].astype(int).astype(str), drop_first=True).astype(float)

# ① 回帰調整：共変量を入れて AI の係数を読む
XA = X.copy(); XA['AI'] = S['AI'].values
係数 = LinearRegression().fit(XA, S['勉強分']).coef_[list(XA.columns).index('AI')]

# ② 傾向スコア + IPW（第8回・第12回の傾向スコアの別の使い方）
ps = LogisticRegression(max_iter=1000).fit(X, S['AI']).predict_proba(X)[:, 1]
a, y = S['AI'].values, S['勉強分'].values
w = ps[a==0] / (1 - ps[a==0])
ipw = y[a==1].mean() - (y[a==0] * w).sum() / w.sum()

print(f'層別        : {層別ATT:+.1f} 分/週')
print(f'回帰調整    : {係数:+.1f} 分/週')
print(f'傾向スコアIPW: {ipw:+.1f} 分/週')
print('\n→ どのやり方でも +24〜33分の範囲。結論の向きと大きさは、手法にあまり左右されない。')

> ### 🧪 発展（読むだけ）：手法を回しただけでは、答えにならない
>
> 実は、この同じデータに **第12回とまったく同じ 1対1 最近傍マッチング**を素直に当てはめると、
> **+186.8 分/週** という、まるで違う答えが出ます。しかも共変量のバランスは**完璧に見えます**（理工系比率：処置 0.455 / マッチ後の対照 0.455）。
>
> 原因は、共変量が全部カテゴリだからです。傾向スコアの値は **177 種類**しかなく、同じ値の人が大量にいる。
> その結果、**対照 7,940 人のうち、たった 156 人が何度も使い回されて**しまい、たまたま勉強時間の短い人が繰り返し選ばれた ── これが人工的な「効果」の正体です。
>
> 📌 **バランスが取れているように見えても、推定が壊れていることがある。**
> だから今日は、素直で確かめやすい**層別**を使い、さらに**回帰・IPW でも同じ結論になるか**を見ました。
> 「ライブラリを呼べば因果が出る」わけではありません。**何を比べているのかを、自分で説明できること**が先です。


## 8. それでも、これは「確証」ではない

層別しても +27 分の差が残りました。「AIを使うと週に27分よけいに勉強する」と書きたくなります。**書けません。** なぜか。

1. **逆の因果かもしれない（逆因果）**
   AIを使うから勉強するのではなく、**もともとよく勉強する学生ほど、課題や論文でAIに触れる機会が多い**のかもしれません。
   このデータは**ある一時点**を切り取ったもの（断面データ）で、**どちらが先か**が分かりません。第12回で「確証には**時間方向**の情報が要る」と言ったのは、まさにこれです。

2. **調整できたのは、たった5つの変数だけ（残差交絡）**
   学年・学部・性別・バイト・住まいはそろえました。でも「**もともとの勉強熱心さ**」「学力」「所属研究室」…はそろえていません。
   そして「勉強熱心さ」こそ、AI利用と勉強時間の**両方に効きそう**な、いちばん怪しい交絡です。

3. **自己申告である**
   勉強時間もAI利用も、本人の申告です。よく勉強する人が多めに答える、AI利用を控えめに答える、といった偏りがあれば結果は動きます。

4. **この 25,340 人は日本の大学生ではない（選択バイアス）**
   §2 で見たとおり、生協を通じた調査で、報告書とも一致しません。**「調査に答えた 25,340 人について」**の話です。

> ## 📌 今日の到達点
> **探索**：AI利用群の勉強時間は長い（+42.4分/週）。**符号は素朴な予想と逆**。
> **層別**：医歯薬系だけ符号が逆（−39.6分）。全体の符号を部分に当てはめてはいけない。
> **調整**：構成の違いを除いても +27分（回帰・IPWでも +24〜33分）。
> **確証？**：**していない**。逆因果・残差交絡・自己申告・選択バイアスが残る。
>
> 第11回から第13回まで、ずっと同じことを練習してきました。
> **「データから言えること」と「言いたいこと」を、区別して書く。** これが、この授業で持ち帰ってほしい唯一のことです。


## 9. まとめと演習

### 演習1（読み取りを書く）✍️
仕送り（`q17_1`）の月額を、**自宅生（`q1i`=1）と自宅外生（`q1i`=2）**で比べなさい。

1. それぞれの**平均**と**人数（母数）**を出す。
2. 結果を**日本語の1〜2文**で書く。**母数と単位を必ず入れる**こと。
   （前回いちばん多かった △ が、この「読み取りを書いていない」でした。数値だけで終わらせない。）

> ⚠️ ヒント：欠損の扱いを、§3 の教訓に従って自分で決め、**何人を外したか**も書くこと。

### 演習2（検算する）🔎
`readme` には「`q17_6`（収入合計）は `q17_1`〜`q17_5` を合計したもの」と書かれています。**本当か確かめなさい。**

1. `q17_1`〜`q17_5` と `q17_6` がすべて揃っている人だけを取り出す。
2. 内訳の和と `q17_6` が**一致する割合**を計算し、`True` の割合を出力する。
3. 一致したなら、それは何を意味するか（そして、**なぜ収入合計と内訳を一緒に相関や回帰に入れてはいけないのか**）を1文で書く。

> 前回、値を出したのに**基本数表との一致確認をしなかった**解答が7件ありました。
> 検算は「正しさの確認」であると同時に、**自分のミスを見つける唯一の方法**です（CSV を読み間違えたまま気づかなかった人もいました）。

### 演習3（結論を書く）✍️
今日の分析の結論を、**3〜5文**で書きなさい。次の4点を必ず含めること。

1. **どんな関連が見られたか**（数値・母数・単位つき）
2. **層別で何が分かったか**（符号が逆転する集団があること）
3. **因果は言えないこと**と、その理由を最低2つ（逆因果／残差交絡／自己申告／選択バイアス から）
4. **出典**：全国大学生活協同組合連合会「第60回学生生活実態調査」、SSJDA 寄託データ、`https://doi.org/10.34500/SSJDA.1794`

> 前回の記述問題で満点を逃した人の**ほぼ全員**が、落としたのは④の**出典**でした。
> 数字は、どこから来たかを書いて初めて「使える数字」になります。


In [ ]:
# 演習1：ここに書く


In [ ]:
# 演習2：ここに書く


**演習3の解答欄**（ここに3〜5文で書く）

> 

---

> 🗑️ **授業のあとに**：`1794.csv` など今日のデータは、指示に従って**手元と Colab の両方から**削除してください。
> 提出する ipynb には**データそのものを貼り付けない**こと（出力の表や数値は構いません）。
